# Statistical Business Analysis of Customer Churn
**Dataset:** 500 customer records. The available fields describe customer tenure, charges, contract/payment attributes and churn. Sales, marketing spend and revenue fields are not present, so the analysis is adapted to churn.

In [ ]:
# Statistical Business Analysis of Customer Churn
# Run cells from top to bottom. Place business_data.csv beside this notebook.
import pandas as pd, numpy as np
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.proportion import proportion_confint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("business_data.csv")
df["ChurnLabel"] = df["Churn"].map({0:"No", 1:"Yes"})
print("Shape:", df.shape)
display(df.head())
display(df.isna().sum().to_frame("Missing values"))

# 1. Descriptive statistics
numeric = ["Tenure","MonthlyCharges","TotalCharges","SeniorCitizen","Churn"]
display(df[numeric].describe().T)
for col in df.select_dtypes(include="object").columns:
    print(f"Mode of {col}: {df[col].mode().iloc[0]}")
print("Numeric modes:", {c: df[c].mode().iloc[0] for c in numeric})

# 2. Distribution and normality
df[["Tenure","MonthlyCharges","TotalCharges"]].hist(bins=20, figsize=(12,4))
plt.tight_layout(); plt.show()
for col in ["Tenure","MonthlyCharges","TotalCharges"]:
    stat, p = stats.shapiro(df[col].sample(min(500,len(df)), random_state=42))
    print(col, "Shapiro-Wilk statistic=", round(stat,4), "p=", p,
          "=> reject normality" if p < .05 else "=> insufficient evidence against normality")

# 3. Pearson correlation and heatmap
corr = df[numeric].corr(method="pearson")
display(corr)
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Pearson correlation heatmap"); plt.tight_layout(); plt.show()

# 4. Hypothesis tests: Welch t-tests, ANOVA, chi-square
churned, retained = df[df.Churn==1], df[df.Churn==0]
for col in ["Tenure","MonthlyCharges","TotalCharges"]:
    result = stats.ttest_ind(churned[col], retained[col], equal_var=False)
    print(f"Welch t-test {col}: t={result.statistic:.4f}, p={result.pvalue:.6g}")
f,p = stats.f_oneway(*[g.MonthlyCharges for _,g in df.groupby("Contract")])
print("ANOVA MonthlyCharges by Contract: F=",f,"p=",p)
ct = pd.crosstab(df["Contract"],df["Churn"])
chi2,p,dof,expected = stats.chi2_contingency(ct)
print("Chi-square Contract x Churn: chi2=",chi2,"df=",dof,"p=",p)
ct

# 5. Confidence intervals
def mean_ci(x, confidence=.95):
    x=np.asarray(x); n=len(x); m=x.mean()
    margin=stats.t.ppf((1+confidence)/2,n-1)*stats.sem(x)
    return m, m-margin, m+margin, margin
for col in ["Tenure","MonthlyCharges","TotalCharges"]:
    print(col, "mean, lower, upper, margin =", mean_ci(df[col]))
n=int(df.Churn.sum()); N=len(df)
print("Churn proportion 95% Wilson CI:", proportion_confint(n,N,method="wilson"))

# 6. Multiple linear regression: model total charges using tenure and monthly charges
ols = smf.ols("TotalCharges ~ Tenure + MonthlyCharges", data=df).fit()
print(ols.summary())

# 7. Logistic regression for binary churn outcome (holdout performance)
X = pd.get_dummies(df[["Tenure","MonthlyCharges","TotalCharges","Contract","PaymentMethod",
                       "PaperlessBilling","SeniorCitizen"]], drop_first=True, dtype=float)
y = df["Churn"]
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
model = Pipeline([("scale",StandardScaler()),
                  ("logit",LogisticRegression(max_iter=2000,class_weight="balanced"))])
model.fit(X_train,y_train)
pred=model.predict(X_test); prob=model.predict_proba(X_test)[:,1]
print("Accuracy:",accuracy_score(y_test,pred),"ROC-AUC:",roc_auc_score(y_test,prob))
print(confusion_matrix(y_test,pred))
print(classification_report(y_test,pred,zero_division=0))

# Business interpretation: association is not proof of causation.
